## import modules

In [ ]:
%pip install torch==2.2
%pip install torchvision==0.17
%pip install matplotlib==3.5.2

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt

## define model architecture

In [ ]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        self.cn1 = nn.Conv2d(1, 16, 3, 1)
        self.cn2 = nn.Conv2d(16, 32, 3, 1)
        self.dp1 = nn.Dropout2d(0.10)
        self.dp2 = nn.Dropout2d(0.25)
        self.fc1 = nn.Linear(4608, 64) # 4608 is basically 12 X 12 X 32
        self.fc2 = nn.Linear(64, 10)
 
    def forward(self, x):
        x = self.cn1(x)
        x = F.relu(x)
        x = self.cn2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dp1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dp2(x)
        x = self.fc2(x)
        op = F.log_softmax(x, dim=1)
        return op

## define training and inference routines

In [ ]:
def train(model, device, train_dataloader, optim, epoch):
    model.train()
    for b_i, (X, y) in enumerate(train_dataloader):
        X, y = X.to(device), y.to(device)
        optim.zero_grad()
        pred_prob = model(X)
        loss = F.nll_loss(pred_prob, y) # nll is the negative likelihood loss
        loss.backward()
        optim.step()
        if b_i % 10 == 0:
            print('epoch: {} [{}/{} ({:.0f}%)]\t training loss: {:.6f}'.format(
                epoch, b_i * len(X), len(train_dataloader.dataset),
                100. * b_i / len(train_dataloader), loss.item()))


In [ ]:
def test(model, device, test_dataloader):
    model.eval()
    loss = 0
    success = 0
    with torch.no_grad():
        for X, y in test_dataloader:
            X, y = X.to(device), y.to(device)
            pred_prob = model(X)
            loss += F.nll_loss(pred_prob, y, reduction='sum').item()  # loss summed across the batch
            pred = pred_prob.argmax(dim=1, keepdim=True)  # us argmax to get the most likely prediction
            success += pred.eq(y.view_as(pred)).sum().item()

    loss /= len(test_dataloader.dataset)

    print('\nTest dataset: Overall Loss: {:.4f}, Overall Accuracy: {}/{} ({:.0f}%)\n'.format(
        loss, success, len(test_dataloader.dataset),
        100. * success / len(test_dataloader.dataset)))


## create data loaders

In [ ]:
# The mean and standard deviation values are calculated as the mean of all pixel values of all images in the training dataset
train_dataloader = torch.utils.data.DataLoader(
    datasets.MNIST('../data', train=True, download=True,
                   transform=transforms.Compose([
                       transforms.ToTensor(),
                       transforms.Normalize((0.1302,), (0.3069,))])), # train_X.mean()/256. and train_X.std()/256.
    batch_size=32, shuffle=True)

test_dataloader = torch.utils.data.DataLoader(
    datasets.MNIST('../data', train=False, 
                   transform=transforms.Compose([
                       transforms.ToTensor(),
                       transforms.Normalize((0.1302,), (0.3069,)) 
                   ])),
    batch_size=500, shuffle=False)

## define optimizer and run training epochs

In [ ]:
torch.manual_seed(0)
device = torch.device("cuda")

model = ConvNet().to(device)
optimizer = optim.Adadelta(model.parameters(), lr=0.5)

## model training

In [ ]:
for epoch in range(1, 3):
    train(model, device, train_dataloader, optimizer, epoch)
    test(model, device, test_dataloader)

## run inference on trained model

In [ ]:
test_samples = enumerate(test_dataloader)
b_i, (sample_data, sample_targets) = next(test_samples)

plt.imshow(sample_data[0][0], cmap='gray', interpolation='none')
plt.show()

In [ ]:
sample_data = sample_data.to(device) 
print(f"Model prediction is : {model(sample_data).data.max(1)[1][0]}")
print(f"Ground truth is : {sample_targets[0]}")

In [ ]:
model_path = '../model/mnist_001.pth'
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")

## LSH for this model

In [ ]:
import hashlib
import os
import json
import random
import time

data_path = "../data/MNIST/raw/t10k-images-idx3-ubyte"

def calculate_file_hash(file_path: str, chunk_size: int = 65536) -> str:
    """
    Purpose: Perform streaming SHA-256 hashing on a large file to generate a Data Hash.
    Args: file_path - path of the file to hash
    Returns: SHA-256 hash string
    """
    sha256 = hashlib.sha256()
    try:
        with open(file_path, 'rb') as f:
            # Read the file chunk by chunk to avoid memory overflow
            while True:
                chunk = f.read(chunk_size)
                if not chunk:
                    break
                sha256.update(chunk)
        return sha256.hexdigest()
    except FileNotFoundError:
        return "ERROR_FILE_NOT_FOUND"

# Example call:
# data_hash = calculate_file_hash("/path/to/local_tuning_data.pt") 
# print(f"Data Hash: {data_hash}")

def generate_lsh(hmi_id: str, data_hash: str, rng_seed: int, hardware_config: dict) -> str:
    """
    Purpose: Combine all metadata and generate the final local identity signature (LSH).
    """
    # 1. Freeze environment parameters (Hardware Config)
    # To generate a deterministic hash, all configuration dictionaries need to be converted into a hashable string.
    # This is a key step to ensure the generated LSH differs across environments.
    hardware_str = json.dumps(hardware_config, sort_keys=True)

    # 2. Construct the seed string (The Seed String)
    seed_string = f"{hmi_id}|{data_hash}|{rng_seed}|{hardware_str}"
    print(seed_string)

    # 3. Generate the final SHA-256 hash
    lsh = hashlib.sha256(seed_string.encode('utf-8')).hexdigest()
    return lsh

## OLC for this model

In [ ]:
# It is recommended to hash the training data; if it does not exist, fall back to the current data_path in this script
train_data_path = "../data/MNIST/raw/train-images-idx3-ubyte"
hash_target_path = train_data_path if os.path.exists(train_data_path) else data_path

# Keep consistent with the current training code
rng_seed = 0  # because torch.manual_seed(0) was used above

# Use the current model filename as the local model ID; you may replace it with another identifier
hmi_id = "mnist_001"

# Hardware/environment information, used as part of the LSH input
hardware_config = {
    "device": str(device),
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
}

# Compute Data Hash and LSH
data_hash = calculate_file_hash(hash_target_path)
local_lsh = generate_lsh(
    hmi_id=hmi_id,
    data_hash=data_hash,
    rng_seed=rng_seed,
    hardware_config=hardware_config
)

# Structured metadata
hmi_raw = "mnist, DL/NN"
goid_raw = "GL(270.4, -28.9, ly26000, SIE-LG-VC)"
te_raw = "Anthropocene, AD2024"

olc = f"M({hmi_raw})−{goid_raw}−TE({te_raw})−LSH({local_lsh})"

metadata = {
    "signature_schema": "OLC-v1",
    "OLC": olc,
    "HMI": {
        "raw": hmi_raw,
        "mother_id": "mnist",
        "family": "DL/NN",
    },
    "GOID": {
        "raw": goid_raw,
    },
    "TE": {
        "raw": te_raw,
        "epoch": "Anthropocene",
        "year": "AD2024",
    },
    "LSH": local_lsh,
    "data_hash": data_hash,
    "hash_target_path": hash_target_path,
    "rng_seed": rng_seed,
    "hardware_config": hardware_config,
    "model_name": model.__class__.__name__,
}

# Save as a checkpoint with metadata instead of only the raw state_dict
model_path = "../model/mnist_001_with_metadata.pth"
os.makedirs(os.path.dirname(model_path), exist_ok=True)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "metadata": metadata,
}

torch.save(checkpoint, model_path)

{
    "saved_to": model_path,
    "OLC": metadata["OLC"],
    "LSH": metadata["LSH"],
}